# Model Evaluation

This notebook evaluates trained models, visualizes confusion matrix, ROC curve, and feature importances.

In [ ]:
import pandas as pd
import numpy as np
import yaml
import joblib
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.metrics import roc_curve, auc, precision_recall_curve
import matplotlib.pyplot as plt
import seaborn as sns

# Load configuration
cfg = yaml.safe_load(open("../config.yaml"))

# Load test data
test = pd.read_csv("../data/test.csv")
text_col = "cleaned_text" if "cleaned_text" in test.columns else "text"
X_test = test[text_col].fillna("")
y_test = test["spam"]

# Load vectorizer and trained models
vectorizer = joblib.load("../models/vectorizer.pkl")
X_test_vec = vectorizer.transform(X_test)

# Load models
models = {}
for model_name in ["naive_bayes", "logistic_regression"]:
    try:
        models[model_name] = joblib.load(f"../models/{model_name}.pkl")
    except:
        pass

print("=" * 60)
print("MODEL EVALUATION RESULTS")
print("=" * 60)

# Evaluate each model
for name, model in models.items():
    y_pred = model.predict(X_test_vec)
    y_proba = model.predict_proba(X_test_vec)[:, 1]
    
    print(f"\n{'='*40}")
    print(f"Model: {name.replace('_', ' ').title()}")
    print(f"{'='*40}")
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
    print(f"F1-Score: {f1_score(y_test, y_pred):.4f}")
    print(f"\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=["Ham", "Spam"]))

# Plot confusion matrix for best model
best_model = max(models.items(), key=lambda x: accuracy_score(y_test, x[1].predict(X_test_vec)))[1]
y_pred_best = best_model.predict(X_test_vec)

fig, ax = plt.subplots(figsize=(5, 4))
cm = confusion_matrix(y_test, y_pred_best)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", 
            xticklabels=["Ham", "Spam"], yticklabels=["Ham", "Spam"])
plt.title("Confusion Matrix - Best Model")
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.tight_layout()
plt.savefig("../reports/confusion_matrix.png", dpi=150)
plt.show()

# ROC Curve comparison
plt.figure(figsize=(8, 6))
for name, model in models.items():
    y_proba = model.predict_proba(X_test_vec)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{name} (AUC={roc_auc:.3f})")

plt.plot([0, 1], [0, 1], "k--", label="Random")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend()
plt.tight_layout()
plt.savefig("../reports/roc_curve.png", dpi=150)
plt.show()

print("\n✓ Evaluation complete! Plots saved to /reports/")